In [5]:
%reload_ext autoreload
%autoreload 2

from tqdm import trange
from flygym.compose import ActuatorType
from miniproject.simulation import MiniprojectSimulation
from submission.controller import Controller

sim = MiniprojectSimulation(level=3, seed=42)
controller = Controller(sim)

for _ in trange(70000):
    joint_angles, adhesion = controller.step(sim)
    sim.set_actuator_inputs(sim.fly.name, ActuatorType.POSITION, joint_angles)
    sim.set_actuator_inputs(sim.fly.name, ActuatorType.ADHESION, adhesion)
    sim.step()
    sim.render_as_needed()

sim.renderer.show_in_notebook()

KeyboardInterrupt: 

In [1]:
import numpy as np
import tqdm
from flygym.compose import ActuatorType
from miniproject import MiniprojectSimulation
from submission.controller import Controller

MAX_NUM_STEPS = 100000
NUM_SEEDS = 10
LEVEL = 4  # change as needed
SUCCESS_RADIUS = 3.0

def run_single_seed(seed):
    sim = MiniprojectSimulation(level=LEVEL, seed=seed, back_cam=False, top_cam=False)
    controller = Controller(sim)

    def get_distance():
        fly_xy = np.array(sim.get_body_positions(sim.fly.name)[0][:2])
        banana_xy = np.array(sim.world.banana_xy)
        return np.linalg.norm(fly_xy - banana_xy)

    for step in range(MAX_NUM_STEPS):
        dist = get_distance()
        if dist <= SUCCESS_RADIUS:
            print(f"  seed={seed} → reached banana in {step} steps (dist={dist:.2f})")
            return dist

        joint_angles, adhesion_signals = controller.step(sim)
        sim.set_actuator_inputs(sim.fly.name, ActuatorType.POSITION, joint_angles)
        sim.set_actuator_inputs(sim.fly.name, ActuatorType.ADHESION, adhesion_signals)
        sim.step()

    final_dist = get_distance()
    print(f"  seed={seed} → timed out (final dist={final_dist:.2f})")
    return final_dist


distances = []
seeds = np.random.randint(0, 101, size=NUM_SEEDS)
for seed in tqdm.tqdm(seeds):
    d = run_single_seed(int(seed))
    distances.append(d)

distances = np.array(distances)
print(f"\n--- Results over {NUM_SEEDS} seeds ---")
print(f"  Mean final distance : {distances.mean():.2f}")
print(f"  Std                 : {distances.std():.2f}")
print(f"  Min / Max           : {distances.min():.2f} / {distances.max():.2f}")
print(f"  Success rate (≤{SUCCESS_RADIUS}mm): {(distances <= SUCCESS_RADIUS).mean()*100:.0f}%")

 10%|█         | 1/10 [00:39<05:51, 39.01s/it]

  seed=14 → timed out (final dist=30.14)


 20%|██        | 2/10 [01:08<04:27, 33.39s/it]

  seed=5 → timed out (final dist=27.73)


 30%|███       | 3/10 [01:44<04:01, 34.46s/it]

...

  seed=81 → timed out (final dist=33.59)


 90%|█████████ | 9/10 [04:55<00:31, 31.43s/it]

  seed=96 → timed out (final dist=26.65)


100%|██████████| 10/10 [05:32<00:00, 33.21s/it]

  seed=79 → timed out (final dist=31.31)

--- Results over 10 seeds ---
  Mean final distance : 29.72
  Std                 : 2.95
  Min / Max           : 25.49 / 33.59
  Success rate (≤3.0mm): 0%
